# narrated-video：CogVideoX 图生视频

这是可选的外部素材生成步骤。先在本地用 `prepare_video_jobs.py` 生成并下载 `video_jobs.zip`，上传后运行全部单元格；完成后下载 `cogvideo-output.zip`，再用 `import_generated_videos.py` 导回项目。免费 Colab 的 GPU、时长和模型可用性不作保证，生成失败时回退到图片运镜。

In [ ]:
!pip -q install -U diffusers transformers accelerate imageio imageio-ffmpeg
import json, os, shutil, zipfile, glob
from pathlib import Path
from google.colab import files
uploaded = files.upload()
bundle = next((name for name in uploaded if name.endswith('.zip')), None)
assert bundle, '请上传 video_jobs.zip'
WORK = Path('/content/cogvideox-work')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()
with zipfile.ZipFile(bundle) as z: z.extractall(WORK)
jobs = json.loads((WORK / 'video_jobs.json').read_text(encoding='utf-8'))['jobs']
len(jobs), jobs[0] if jobs else None

In [ ]:
import torch
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image
MODEL = 'THUDM/CogVideoX-5b-I2V'
pipe = CogVideoXImageToVideoPipeline.from_pretrained(MODEL, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
OUTPUT = WORK / 'output'
OUTPUT.mkdir(exist_ok=True)
print('Model loaded:', MODEL)

In [ ]:
for job in jobs:
    image = Image.open(WORK / job['image']).convert('RGB')
    prompt = job['prompt'] + ' Constraints: ' + '; '.join(job.get('constraints', []))
    generator = torch.Generator(device='cuda').manual_seed(int(job['seed']))
    requested_frames = max(13, min(49, round(float(job['duration_target']) * 8) + 1))
    num_frames = 1 + 4 * ((requested_frames - 1) // 4)
    result = pipe(image=image, prompt=prompt, num_frames=num_frames, guidance_scale=6,
                  num_inference_steps=50, generator=generator)
    output = WORK / job['output']
    output.parent.mkdir(exist_ok=True)
    export_to_video(result.frames[0], str(output), fps=8)
    print(job['id'], output)

In [ ]:
output_zip = Path('/content/cogvideo-output.zip')
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(WORK / 'video_jobs.json', 'video_jobs.json')
    for path in OUTPUT.glob('*.mp4'):
        z.write(path, 'output/' + path.name)
files.download(str(output_zip))